In [107]:
# --- Cell 1: Import our Tools ---
import os
from dotenv import load_dotenv
import os.path
import base64
from email.message import EmailMessage
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# We are using Google's rock-solid Gemini AI!
import google.generativeai as genai
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

# Tell Python to unlock and read the .env file right now
load_dotenv()
SCOPES = ['https://www.googleapis.com/auth/gmail.send']
tasks = [] 
print("Cell 1 Complete: Tools loaded and memory is ready! ✅")

Cell 1 Complete: Tools loaded and memory is ready! ✅


In [108]:
# --- Cell 2: Define our Robot's Abilities ---

# 1. Connect to Gemini (Paste your Gemini API Key here!)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

# 2. Load the Gemini 1.5 Flash model
model = genai.GenerativeModel('models/gemini-flash-latest')

def generate_email_draft(task_name, recipient_email):
    # Grab the part of the email before the @ symbol to use as the name
    client_name = "client"
    
    print(f"🧠 Gemini AI is drafting a custom email for: '{client_name}'...")
    
    # We tell the AI to address the email specifically to this client
    prompt = f"Write a short, professional follow-up email addressed to '{client_name}' regarding this task: '{task_name}'. Do not include a Subject line. Sign the email as 'Your Market Manager'."
    
    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        return f"Gemini Error: {e}"

def send_email_api(draft_text, recipient_email):
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    try:
        service = build('gmail', 'v1', credentials=creds)
        message = EmailMessage()
        message.set_content(draft_text)
        message['To'] = recipient_email
        message['From'] = 'me'
        message['Subject'] = 'Task Reminder'

        encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()
        create_message = {'raw': encoded_message}

        send_message = service.users().messages().send(userId="me", body=create_message).execute()
        print(f"Email sent successfully! 🚀 (Message ID: {send_message['id']})")
    except Exception as error:
        print(f"Oops! Error: {error}")

print("Cell 2 Complete: Robot's brain is updated to Google Gemini! 🧠")

Cell 2 Complete: Robot's brain is updated to Google Gemini! 🧠


In [109]:
# --- Cell 3: Add Tasks to Memory

my_new_task = ["Ads performance is down. Please follow up with the client and offer assistance to improve results."]
my_due_date = ["2026-04-20"]

for i in range(len(my_new_task)):
    tasks.append({"name": my_new_task[i], "due_date": my_due_date[i]})

print(f"Added tasks! You now have {len(tasks)} tasks in memory. 📝")

Added tasks! You now have 1 tasks in memory. 📝


In [110]:
# --- Cell 4: Draft and Send! ---

# 1. Get the last task we added
current_task = tasks[-1]

# 2. Our List of targets
target_emails = ["mzirayjohn4@gmail.com", "john@nm-aist.ac.tz"]

# 3. The Loop: Write a custom draft AND send it individually!
for email in target_emails:
    print(f"\n-----------------------------------------")
    print(f"Attempting to send to {email}...")
    
    # Write a custom draft FOR this specific email
    custom_draft = generate_email_draft(current_task["name"], email)
    
    print("\n--- Here is the custom draft ---")
    print(custom_draft)
    
    # Send the custom email
    send_email_api(custom_draft, email)

print("\nAll customized emails have been sent! 🚀")


-----------------------------------------
Attempting to send to mzirayjohn4@gmail.com...
🧠 Gemini AI is drafting a custom email for: 'client'...

--- Here is the custom draft ---
Dear Client,

I have been reviewing your recent ad performance and noticed a dip in results. I would like to offer my assistance to help optimize your campaigns and get performance back on track.

Are you available for a brief call this week to discuss some strategic adjustments and improvements?

Best regards,

Your Market Manager
Email sent successfully! 🚀 (Message ID: 19d888b42259e7c0)

-----------------------------------------
Attempting to send to john@nm-aist.ac.tz...
🧠 Gemini AI is drafting a custom email for: 'client'...

--- Here is the custom draft ---
Dear Client,

I’ve been monitoring your recent ad performance and noticed that results have trended downward. 

I would like to offer my assistance in reviewing your current strategy and identifying specific areas where we can optimize to improve your